In [ ]:
from pathlib import Path
import sys
import xml.etree.ElementTree as ET
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Derive the root of this package
package_root = Path.cwd().resolve().parents[2]

# Include this pakage
if str(package_root) not in sys.path:
    sys.path.append(str(package_root))

from hamageolib.utils.geometry_utilities import haversine_distance

In [ ]:
# ============================================================
# Input files
# ============================================================

gebco_file = Path(
    "/home/lochy/Dataset/GEBCO/GEBCO_26_Aug_2026_SA_extracted/gebco_2026_n-0.967_s-60.205_w-109.688_e-24.609.nc"
)

kml_file = Path(
    # "/home/lochy/Dataset/GEBCO/Path_26_Aug_2026_SA.kml"
    "/home/lochy/Dataset/GEBCO/Path_26_Aug_2026_SA_Altiplano.kml"
)

print("Path:", gebco_file)
print("Exists:", gebco_file.exists())
print("Is file:", gebco_file.is_file())
print("Is directory:", gebco_file.is_dir())

if gebco_file.is_dir():
    print("\nContents:")
    for path in gebco_file.iterdir():
        print(path)

In [ ]:
# Spacing of points along the final profile, in km
profile_spacing_km = 1.0


# ============================================================
# Read Google Earth KML path
# ============================================================

def read_kml_path(kml_file):
    """
    Read the first LineString from a Google Earth KML file.

    Returns
    -------
    lons : numpy.ndarray
        Longitudes in degrees.
    lats : numpy.ndarray
        Latitudes in degrees.
    """

    tree = ET.parse(kml_file)
    root = tree.getroot()

    # KML normally uses this namespace
    namespace = {"kml": "http://www.opengis.net/kml/2.2"}

    coordinate_element = root.find(
        ".//kml:LineString/kml:coordinates",
        namespace
    )

    if coordinate_element is None:
        raise ValueError(f"No LineString found in {kml_file}")

    coordinates = []

    for point in coordinate_element.text.strip().split():
        values = point.split(",")

        lon = float(values[0])
        lat = float(values[1])

        coordinates.append((lon, lat))

    coordinates = np.asarray(coordinates)

    return coordinates[:, 0], coordinates[:, 1]


def resample_path(lons, lats, spacing_km=1.0):
    """
    Resample a lon/lat path at approximately constant spacing.

    Linear interpolation in lon/lat is used within each segment.
    """

    output_lons = [lons[0]]
    output_lats = [lats[0]]
    output_distances = [0.0]

    cumulative_distance_km = 0.0

    for i in range(len(lons) - 1):

        lon0, lat0 = lons[i], lats[i]
        lon1, lat1 = lons[i + 1], lats[i + 1]

        segment_distance_km = haversine_distance(
            lon0, lat0,
            lon1, lat1
        )

        distances = np.arange(
            spacing_km,
            segment_distance_km,
            spacing_km
        )

        for distance_km in distances:

            fraction = distance_km / segment_distance_km

            lon = lon0 + fraction * (lon1 - lon0)
            lat = lat0 + fraction * (lat1 - lat0)

            output_lons.append(lon)
            output_lats.append(lat)
            output_distances.append(
                cumulative_distance_km + distance_km
            )

        cumulative_distance_km += segment_distance_km

        output_lons.append(lon1)
        output_lats.append(lat1)
        output_distances.append(cumulative_distance_km)

    return (
        np.asarray(output_lons),
        np.asarray(output_lats),
        np.asarray(output_distances)
    )


# ============================================================
# Read path
# ============================================================

path_lons, path_lats = read_kml_path(kml_file)

print("Original Google Earth path:")
print(f"  Number of vertices: {len(path_lons)}")
print(f"  Longitude range: {path_lons.min():.4f} - {path_lons.max():.4f}")
print(f"  Latitude range:  {path_lats.min():.4f} - {path_lats.max():.4f}")


# ============================================================
# Resample path
# ============================================================

profile_lons, profile_lats, distances_km = resample_path(
    path_lons,
    path_lats,
    spacing_km=profile_spacing_km
)

print()
print("Resampled profile:")
print(f"  Number of points: {len(profile_lons)}")
print(f"  Total length: {distances_km[-1]:.2f} km")


# ============================================================
# Read GEBCO
# ============================================================

ds = xr.open_dataset(gebco_file)

print()
print("GEBCO dataset:")
print(ds)


# ============================================================
# Interpolate GEBCO elevation along path
# ============================================================

lon_points = xr.DataArray(
    profile_lons,
    dims="profile"
)

lat_points = xr.DataArray(
    profile_lats,
    dims="profile"
)

elevation = ds["elevation"].interp(
    lon=lon_points,
    lat=lat_points
).values


# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    distances_km,
    elevation,
    linewidth=1.5
)

# Sea level
ax.axhline(
    0.0,
    linewidth=0.8,
    linestyle="--"
)

ax.set_xlabel("Distance along profile (km)")
ax.set_ylabel("Elevation (m)")
ax.set_title("GEBCO topography/bathymetry profile")

ax.grid(True, alpha=0.3)

fig.tight_layout()

plt.show()